<a href="https://colab.research.google.com/github/FaizaTasneeem/apachejit-defect-prediction/blob/main/apachejit_defect_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install catboost shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 35.8 MB/s  0:00:02


In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/hosseinkshvrz/apachejit/master/dataset/apachejit_total.csv"
df = pd.read_csv(url)
buggy = df[df['buggy'] == True]

print(f"Columns: {', ' .join(df.columns)}")
print(f"Row count: {len(df)}")
print(f"Share of buggy commits: {round((len(buggy)/len(df)*100), 2)} %")


Columns: commit_id, project, buggy, fix, year, author_date, la, ld, nf, nd, ns, ent, ndev, age, nuc, aexp, arexp, asexp
Row count: 106674
Share of buggy commits: 26.47 %


In [ ]:
print(df.dtypes)

commit_id       object
project         object
buggy             bool
fix               bool
year             int64
author_date      int64
la               int64
ld               int64
nf               int64
nd               int64
ns               int64
ent            float64
ndev           float64
age            float64
nuc            float64
aexp             int64
arexp          float64
asexp          float64
dtype: object


In [ ]:
df["author_date"] = pd.to_datetime(df["author_date"], unit="s")
print("Earliest commit:", df["author_date"].min())
print("Latest commit:", df["author_date"].max())

Earliest commit: 2003-09-11 14:54:46
Latest commit: 2019-12-26 17:39:53


In [ ]:
print("Checking if author_date is sorted in ascending order: ", df["author_date"].is_monotonic_increasing)
print("Checking if author_date is sorted in descending order: ", df["author_date"].is_monotonic_decreasing)

Checking if author_date is sorted in ascending order:  False
Checking if author_date is sorted in descending order:  False


In [ ]:
print(df.isnull().sum())
print("Total missing:", df.isnull().sum().sum())

commit_id      0
project        0
buggy          0
fix            0
year           0
author_date    0
la             0
ld             0
nf             0
nd             0
ns             0
ent            0
ndev           0
age            0
nuc            0
aexp           0
arexp          0
asexp          0
dtype: int64
Total missing: 0


In [ ]:
print("Duplicate commit_ids:", df["commit_id"].duplicated().sum())

Duplicate commit_ids: 0


In [ ]:
print("Number of project values:", df["project"].nunique())
print("Number of commits per project:", df["project"].value_counts())

Number of project values: 15
Number of commits per project: project
apache/camel               22700
apache/ignite              12036
apache/hadoop              11964
apache/flink               11691
apache/hbase                8730
apache/cassandra            8159
apache/groovy               8059
apache/hive                 6842
apache/activemq             6126
apache/hadoop-hdfs          2907
apache/kafka                2384
apache/spark                1465
apache/zeppelin             1451
apache/hadoop-mapreduce     1321
apache/zookeeper             839
Name: count, dtype: int64


In [5]:
features = ["la", "ld", "nf", "nd", "ns", "ent", "ndev", "age", "nuc", "aexp", "arexp", "asexp"]
medians = df.groupby("buggy")[features].median()
print(medians)

          la    ld   nf   nd   ns       ent  ndev        age        nuc  \
buggy                                                                     
False   22.0   6.0  2.0  2.0  1.0  0.918296   2.5  11.166667  13.000000   
True   138.0  27.0  6.0  3.0  1.0  1.768599   2.0  19.500000  22.614458   

        aexp  arexp  asexp  
buggy                       
False  241.0  191.0   52.0  
True   180.0  150.0    5.0  


In [ ]:
# from google.colab import data_table

# start = 0
# data_table.DataTable(df.iloc[start:start + 20000], num_rows_per_page=50)